## Libraries, Packages, and Classes

In [1]:
EPOCHS = 6

In [2]:
!pip install torch transformers datasets peft mlflow evaluate sacrebleu rouge_score -q

In [3]:
!pip install --upgrade torchao peft transformers datasets evaluate sacrebleu -q

In [4]:
import gc
import os
import evaluate
import numpy as np
import pandas as pd
import mlflow
import torch
import mlflow.sklearn

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    MT5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

W0801 20:40:35.699000 10035 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0801 20:40:35.729000 10035 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [5]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [6]:
def flush_gpu_memory():
    """Utility function to immediately free cached CUDA VRAM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [7]:
class NLLBSeq2SeqTrainer:
    def __init__(
        self,
        model_name: str = "facebook/nllb-200-1.3B",
        experiment_name: str = "nllb_ablation_suite",
        src_lang: str = "eng_Latn",
        tgt_lang: str = "swh_Latn"
    ):
        self.model_name = model_name
        self.experiment_name = experiment_name
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.sacrebleu = evaluate.load("sacrebleu")
        mlflow.set_experiment(self.experiment_name)

    def _augment_data(self, dataset: Dataset) -> Dataset:
        aug_sources, aug_targets = [], []
        for item in dataset:
            aug_sources.extend([item["source_text"], f"Translate: {item['source_text']}"])
            aug_targets.extend([item["target_text"], item["target_text"]])
        return Dataset.from_dict({"source_text": aug_sources, "target_text": aug_targets})

    def _prepare_tokenizer_and_model(self, tuning_strategy: str, lora_r: int, lora_alpha: int):
        tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)

        tokenizer.src_lang = self.src_lang
        tokenizer.tgt_lang = self.tgt_lang

        if tuning_strategy == "freeze_encoder":
            for param in model.get_encoder().parameters():
                param.requires_grad = False

        elif tuning_strategy == "lora":
            peft_config = LoraConfig(
                task_type=TaskType.SEQ_2_SEQ_LM,
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=0.1,
                target_modules=["q_proj", "v_proj", "k_proj", "out_proj"]
            )
            model = get_peft_model(model, peft_config)

        return tokenizer, model

    def _save_checkpoint(
        self,
        model,
        tokenizer,
        artifact_path: str,
        tuning_strategy: str
    ):
        """
        Saves only the trainable parameters.

        Returns
        -------
        checkpoint_type : str
            Either 'lora' or 'delta'
        """

        os.makedirs(artifact_path, exist_ok=True)
        tokenizer.save_pretrained(artifact_path)

        # ---------------------------------------------------
        # Zero-shot
        # ---------------------------------------------------
        if tuning_strategy == "zero_shot":
            return "base"

        # ---------------------------------------------------
        # LoRA
        # ---------------------------------------------------
        elif tuning_strategy == "lora":
            model.save_pretrained(artifact_path)
            return "lora"

        # ---------------------------------------------------
        # Freeze encoder
        # ---------------------------------------------------
        elif tuning_strategy == "freeze_encoder":

            model = model.cpu()

            delta = {}

            state_dict = model.state_dict()

            for name, param in model.named_parameters():
                if param.requires_grad:
                    delta[name] = state_dict[name]

            torch.save(
                delta,
                os.path.join(artifact_path, "delta.pt")
            )

            return "delta"

        else:
            raise ValueError(f"Unknown tuning strategy: {tuning_strategy}")

    def _load_checkpoint(
        self,
        artifact_path: str,
        checkpoint_type: str
    ):
        """
        Reconstructs a trained model from a saved checkpoint.

        checkpoint_type:
            - base  : zero-shot model
            - lora  : LoRA adapter weights
            - delta : trainable parameter weights
        """

        tokenizer = AutoTokenizer.from_pretrained(
            artifact_path
        )

        tokenizer.src_lang = self.src_lang
        tokenizer.tgt_lang = self.tgt_lang

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name
        )

        # ---------------------------------------------------
        # Zero-shot: use base model directly
        # ---------------------------------------------------
        if checkpoint_type == "base":
            pass

        # ---------------------------------------------------
        # LoRA adapter
        # ---------------------------------------------------
        elif checkpoint_type == "lora":

            model = PeftModel.from_pretrained(
                model,
                artifact_path
            )

        # ---------------------------------------------------
        # Trainable parameter checkpoint
        # ---------------------------------------------------
        elif checkpoint_type == "delta":

            delta_path = os.path.join(
                artifact_path,
                "delta.pt"
            )

            delta = torch.load(
                delta_path,
                map_location="cpu"
            )

            model.load_state_dict(
                delta,
                strict=False
            )

        else:
            raise ValueError(
                f"Unknown checkpoint type: {checkpoint_type}"
            )

        return tokenizer, model

    def run_experiment(
        self,
        dataset: Dataset,
        use_augmentation: bool = False,
        tuning_strategy: str = "lora",
        lora_r: int = 8,
        lora_alpha: int = 32,
        lr: float = 5e-4,
        epochs: int = 5,
        batch_size: int = 2
    ):
        flush_gpu_memory()
        run_name = f"nllb_{tuning_strategy}_aug={use_augmentation}"

        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({
                "model_name": self.model_name,
                "tuning_strategy": tuning_strategy,
                "use_augmentation": use_augmentation,
                "lora_r": lora_r if tuning_strategy == "lora" else 0,
                "learning_rate": lr,
                "epochs": epochs if tuning_strategy != "zero_shot" else 0,
                "src_lang": self.src_lang,
                "tgt_lang": self.tgt_lang
            })
            print(f"\n================ Executing: {run_name} ================")

            working_ds = self._augment_data(dataset) if use_augmentation else dataset
            tokenizer, model = self._prepare_tokenizer_and_model(tuning_strategy, lora_r, lora_alpha)

            def preprocess_function(examples):
                model_inputs = tokenizer(examples["source_text"], max_length=128, truncation=True)
                labels = tokenizer(text_target=examples["target_text"], max_length=128, truncation=True)
                model_inputs["labels"] = labels["input_ids"]
                return model_inputs

            tokenized_ds = working_ds.map(preprocess_function, batched=True)

            def compute_metrics(eval_preds):
                preds, labels = eval_preds
                if isinstance(preds, tuple):
                    preds = preds[0]
                decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
                labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
                decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
                decoded_labels_formatted = [[label.strip()] for label in decoded_labels]

                bleu_score = self.sacrebleu.compute(
                    predictions=[p.strip() for p in decoded_preds],
                    references=decoded_labels_formatted
                )
                return {"bleu": bleu_score["score"]}

            output_dir = f"./tmp_{run_name}"
            training_args = Seq2SeqTrainingArguments(
                output_dir=output_dir,
                per_device_train_batch_size=batch_size,
                per_device_eval_batch_size=batch_size,
                predict_with_generate=True,
                generation_max_length=128,
                num_train_epochs=epochs if tuning_strategy != "zero_shot" else 0,
                learning_rate=lr,
                logging_steps=1,
                eval_strategy="epoch" if tuning_strategy != "zero_shot" else "no",
                report_to=["mlflow"],
                save_strategy="no"
            )

            trainer = Seq2SeqTrainer(
                model=model,
                args=training_args,
                train_dataset=tokenized_ds,
                eval_dataset=tokenized_ds,
                processing_class=tokenizer,
                data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
                compute_metrics=compute_metrics
            )

            metrics = {}
            if tuning_strategy == "zero_shot":
                eval_res = trainer.evaluate()
                metrics["eval_bleu"] = eval_res["eval_bleu"]
            else:
                train_res = trainer.train()
                eval_res = trainer.evaluate()
                metrics["train_loss"] = train_res.training_loss
                metrics["eval_bleu"] = eval_res["eval_bleu"]

            mlflow.log_metrics(metrics)

            artifact_path = f"./artifacts/{run_name}"

            checkpoint_type = self._save_checkpoint(
                model,
                tokenizer,
                artifact_path,
                tuning_strategy
            )

            mlflow.log_param(
                "checkpoint_type",
                checkpoint_type
            )

            mlflow.log_artifacts(
                artifact_path,
                artifact_path="saved_weights"
            )

            print(f"Completed! Metrics: {metrics}")

            del trainer
            del model
            del tokenizer

            flush_gpu_memory()

            return {
                "artifact_path": artifact_path,
                "checkpoint_type": checkpoint_type,
                "metrics": metrics
            }

    def predict_and_display(
        self,
        strategy,
        artifact_path,
        checkpoint_type,
        dataset: Dataset,
        num_samples: int = 5
    ):
        tokenizer, model = self._load_checkpoint(
            artifact_path,
            checkpoint_type
        )

        model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)

        results = []
        predictions = []
        references = []

        sample_count = min(num_samples, len(dataset))
        forced_bos_id = tokenizer.convert_tokens_to_ids(self.tgt_lang)

        for i in range(sample_count):

            src = dataset[i]["source_text"]
            tgt = dataset[i]["target_text"]

            inputs = tokenizer(
                src,
                return_tensors="pt"
            ).to(device)

            with torch.no_grad():

                tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_id,
                    max_new_tokens=128,
                    max_length=None
                )

            pred = tokenizer.decode(
                tokens[0],
                skip_special_tokens=True
            ).strip()

            predictions.append(pred)
            references.append([tgt.strip()])

            results.append({
                "Source Text": src,
                "Ground Truth": tgt,
                "Model Prediction": pred
            })



        bleu_score = self.sacrebleu.compute(
            predictions=predictions,
            references=references
        )

        bleu = bleu_score["score"]

        # Add BLEU column to every prediction row
        for row in results:
            row["BLEU Score"] = bleu

        df = pd.DataFrame(results)

        print(
            f"\n================ Sample Validation Predictions "
            f"({self.model_name}) ================"
        )

        print(f"Strategy       : {strategy}")
        print(f"Checkpoint     : {checkpoint_type}")
        print(f"Validation BLEU: {bleu:.2f}")

        with pd.option_context(
            'display.max_colwidth',
            None
        ):
            display(df)

        del model
        del tokenizer

        flush_gpu_memory()

        return df, bleu

In [8]:
class MBARTSeq2SeqTrainer:
    def __init__(
        self,
        model_name: str = "facebook/mbart-large-50-many-to-many-mmt",
        experiment_name: str = "mbart_ablation_suite",
        src_lang: str = "en_XX",
        tgt_lang: str = "sw_KE"
    ):
        self.model_name = model_name
        self.experiment_name = experiment_name
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.sacrebleu = evaluate.load("sacrebleu")
        mlflow.set_experiment(self.experiment_name)

    def _augment_data(self, dataset: Dataset) -> Dataset:
        aug_sources = []
        aug_targets = []

        for item in dataset:
            aug_sources.extend([
                item["source_text"],
                f"Translate: {item['source_text']}"
            ])

            aug_targets.extend([
                item["target_text"],
                item["target_text"]
            ])

        return Dataset.from_dict({
            "source_text": aug_sources,
            "target_text": aug_targets
        })

    def _prepare_tokenizer_and_model(
        self,
        tuning_strategy: str,
        lora_r: int,
        lora_alpha: int
    ):

        tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)

        tokenizer.src_lang = self.src_lang
        tokenizer.tgt_lang = self.tgt_lang

        if tuning_strategy == "freeze_encoder":
            for param in model.model.encoder.parameters():
                param.requires_grad = False

        elif tuning_strategy == "lora":
            peft_config = LoraConfig(
                task_type=TaskType.SEQ_2_SEQ_LM,
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=0.1,
                target_modules=[
                    "q_proj",
                    "k_proj",
                    "v_proj",
                    "out_proj"
                ]
            )

            model = get_peft_model(model, peft_config)

        return tokenizer, model

    def _save_checkpoint(
        self,
        model,
        tokenizer,
        artifact_path: str,
        tuning_strategy: str
    ):
        """
        Saves only the trainable parameters.

        Returns
        -------
        checkpoint_type : str
            Either 'lora' or 'delta'
        """

        os.makedirs(artifact_path, exist_ok=True)
        tokenizer.save_pretrained(artifact_path)

        # ---------------------------------------------------
        # Zero-shot
        # ---------------------------------------------------
        if tuning_strategy == "zero_shot":
            return "base"

        # ---------------------------------------------------
        # LoRA
        # ---------------------------------------------------
        elif tuning_strategy == "lora":
            model.save_pretrained(artifact_path)
            return "lora"

        # ---------------------------------------------------
        # Freeze encoder
        # ---------------------------------------------------
        elif tuning_strategy == "freeze_encoder":

            model = model.cpu()

            delta = {}

            state_dict = model.state_dict()

            for name, param in model.named_parameters():
                if param.requires_grad:
                    delta[name] = state_dict[name]

            torch.save(
                delta,
                os.path.join(artifact_path, "delta.pt")
            )

            return "delta"

        else:
            raise ValueError(f"Unknown tuning strategy: {tuning_strategy}")

    def _load_checkpoint(
        self,
        artifact_path: str,
        checkpoint_type: str
    ):
        """
        Reconstructs a trained model from a saved checkpoint.

        checkpoint_type:
            - base  : zero-shot model
            - lora  : LoRA adapter weights
            - delta : trainable parameter weights
        """

        tokenizer = AutoTokenizer.from_pretrained(
            artifact_path
        )

        tokenizer.src_lang = self.src_lang
        tokenizer.tgt_lang = self.tgt_lang

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name
        )

        # ---------------------------------------------------
        # Zero-shot: use base model directly
        # ---------------------------------------------------
        if checkpoint_type == "base":
            pass

        # ---------------------------------------------------
        # LoRA adapter
        # ---------------------------------------------------
        elif checkpoint_type == "lora":

            model = PeftModel.from_pretrained(
                model,
                artifact_path
            )

        # ---------------------------------------------------
        # Trainable parameter checkpoint
        # ---------------------------------------------------
        elif checkpoint_type == "delta":

            delta_path = os.path.join(
                artifact_path,
                "delta.pt"
            )

            delta = torch.load(
                delta_path,
                map_location="cpu"
            )

            model.load_state_dict(
                delta,
                strict=False
            )

        else:
            raise ValueError(
                f"Unknown checkpoint type: {checkpoint_type}"
            )

        return tokenizer, model

    def run_experiment(
        self,
        dataset: Dataset,
        use_augmentation: bool = False,
        tuning_strategy: str = "lora",
        lora_r: int = 8,
        lora_alpha: int = 32,
        lr: float = 5e-4,
        epochs: int = 5,
        batch_size: int = 2
    ):

        flush_gpu_memory()

        run_name = f"mbart_{tuning_strategy}_aug={use_augmentation}"

        with mlflow.start_run(run_name=run_name):

            mlflow.log_params({
                "model_name": self.model_name,
                "tuning_strategy": tuning_strategy,
                "augmentation": use_augmentation,
                "lora_r": lora_r if tuning_strategy == "lora" else 0,
                "learning_rate": lr,
                "epochs": epochs if tuning_strategy != "zero_shot" else 0,
                "src_lang": self.src_lang,
                "tgt_lang": self.tgt_lang
            })

            print(f"\n================ {run_name} ================")

            working_ds = (
                self._augment_data(dataset)
                if use_augmentation
                else dataset
            )

            tokenizer, model = self._prepare_tokenizer_and_model(
                tuning_strategy,
                lora_r,
                lora_alpha
            )

            def preprocess_function(examples):

                model_inputs = tokenizer(
                    examples["source_text"],
                    max_length=128,
                    truncation=True
                )

                labels = tokenizer(
                    text_target=examples["target_text"],
                    max_length=128,
                    truncation=True
                )

                model_inputs["labels"] = labels["input_ids"]

                return model_inputs

            tokenized_ds = working_ds.map(
                preprocess_function,
                batched=True
            )

            def compute_metrics(eval_preds):

                preds, labels = eval_preds

                if isinstance(preds, tuple):
                    preds = preds[0]

                decoded_preds = tokenizer.batch_decode(
                    preds,
                    skip_special_tokens=True
                )

                labels = np.where(
                    labels != -100,
                    labels,
                    tokenizer.pad_token_id
                )

                decoded_labels = tokenizer.batch_decode(
                    labels,
                    skip_special_tokens=True
                )

                bleu = self.sacrebleu.compute(
                    predictions=[x.strip() for x in decoded_preds],
                    references=[[x.strip()] for x in decoded_labels]
                )

                return {"bleu": bleu["score"]}

            training_args = Seq2SeqTrainingArguments(
                output_dir=f"./tmp_{run_name}",
                per_device_train_batch_size=batch_size,
                per_device_eval_batch_size=batch_size,
                predict_with_generate=True,
                generation_max_length=128,
                num_train_epochs=epochs if tuning_strategy != "zero_shot" else 0,
                learning_rate=lr,
                logging_steps=1,
                eval_strategy="epoch" if tuning_strategy != "zero_shot" else "no",
                save_strategy="no",
                report_to=["mlflow"]
            )

            trainer = Seq2SeqTrainer(
                model=model,
                args=training_args,
                train_dataset=tokenized_ds,
                eval_dataset=tokenized_ds,
                processing_class=tokenizer,
                data_collator=DataCollatorForSeq2Seq(
                    tokenizer,
                    model=model
                ),
                compute_metrics=compute_metrics
            )

            metrics = {}

            if tuning_strategy == "zero_shot":

                eval_res = trainer.evaluate()

                metrics["eval_bleu"] = eval_res["eval_bleu"]

            else:

                train_res = trainer.train()

                eval_res = trainer.evaluate()

                metrics["train_loss"] = train_res.training_loss
                metrics["eval_bleu"] = eval_res["eval_bleu"]

            mlflow.log_metrics(metrics)

            artifact_path = f"./artifacts/{run_name}"

            checkpoint_type = self._save_checkpoint(
                model,
                tokenizer,
                artifact_path,
                tuning_strategy
            )

            mlflow.log_param(
                "checkpoint_type",
                checkpoint_type
            )

            mlflow.log_artifacts(
                artifact_path,
                artifact_path="saved_weights"
            )

            del trainer
            del model
            del tokenizer

            flush_gpu_memory()

            return {
                "artifact_path": artifact_path,
                "checkpoint_type": checkpoint_type,
                "metrics": metrics
            }

    def predict_and_display(
        self,
        strategy,
        artifact_path,
        checkpoint_type,
        dataset: Dataset,
        num_samples: int = 5
    ):
        tokenizer, model = self._load_checkpoint(
            artifact_path,
            checkpoint_type
        )

        model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)

        results = []
        predictions = []
        references = []

        sample_count = min(num_samples, len(dataset))
        forced_bos_id = tokenizer.convert_tokens_to_ids(self.tgt_lang)

        for i in range(sample_count):

            src = dataset[i]["source_text"]
            tgt = dataset[i]["target_text"]

            inputs = tokenizer(
                src,
                return_tensors="pt"
            ).to(device)

            with torch.no_grad():

                tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_id,
                    max_new_tokens=128,
                    max_length=None
                )

            pred = tokenizer.decode(
                tokens[0],
                skip_special_tokens=True
            ).strip()

            predictions.append(pred)
            references.append([tgt.strip()])

            results.append({
                "Source Text": src,
                "Ground Truth": tgt,
                "Model Prediction": pred
            })

        bleu_score = self.sacrebleu.compute(
            predictions=predictions,
            references=references
        )

        bleu = bleu_score["score"]

        # Add BLEU column to every prediction row
        for row in results:
            row["BLEU Score"] = bleu

        df = pd.DataFrame(results)

        print(
            f"\n================ Sample Validation Predictions "
            f"({self.model_name}) ================"
        )

        print(f"Strategy       : {strategy}")
        print(f"Checkpoint     : {checkpoint_type}")
        print(f"Validation BLEU: {bleu:.2f}")

        with pd.option_context(
            'display.max_colwidth',
            None
        ):
            display(df)

        del model
        del tokenizer

        flush_gpu_memory()

        return df, bleu

## Preamble

In [9]:
few_shot_data = pd.read_csv('nlp_few_shot.csv')

## Experimental Design

### Simple Ablation Studies on NLLB-200 and mBART

In [10]:
mlflow.set_experiment("nllb_vs_mbart_ablations")

<Experiment: artifact_location='/home/jovyan/DSA4020-Natural-Language-Processing/notebooks/mlruns/1', creation_time=1785615098320, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785615098320, lifecycle_stage='active', name='nllb_vs_mbart_ablations', tags={}, trace_location=None, workspace='default'>

In [11]:
train_df = pd.concat([few_shot_data.iloc[0:19]])

In [12]:
validation_df = pd.concat([few_shot_data.iloc[20:29]])

In [13]:
train_data = Dataset.from_dict({
    "source_text": train_df["English"],
    "target_text": train_df["Swahili"]
})

In [14]:
validation_data = Dataset.from_dict({
    "source_text": validation_df["English"],
    "target_text": validation_df["Swahili"]
})

In [15]:
strategies = ["freeze_encoder", "lora", "zero_shot"]

In [16]:
experiments = {
    "nllb": {},
    "mbart": {}
}

In [17]:
nllb_trainer = NLLBSeq2SeqTrainer(
    model_name="facebook/nllb-200-1.3B",
    experiment_name="nllb_vs_mbart_ablations",
    src_lang="eng_Latn",
    tgt_lang="swh_Latn"
)

In [18]:
for strategy in strategies:

    experiments["nllb"][strategy] = nllb_trainer.run_experiment(
        dataset=train_data,
        tuning_strategy=strategy,
        epochs=EPOCHS,
        lr=5e-4
    )

    flush_gpu_memory()


================ Executing: nllb_freeze_encoder_aug=False ================


Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

Map:   0%|          | 0/19 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu
1,0.057604,0.913106,46.002021
2,0.406575,0.349416,64.126821
3,0.720692,0.110963,82.596859
4,0.329528,0.073798,59.069414
5,0.003967,0.017867,94.863918
6,0.013884,0.027446,96.278847


Training Loss,Validation Loss,Epoch,Bleu
0.013884,0.027446,6,96.278847


...

Completed! Metrics: {'train_loss': 0.5008558905373017, 'eval_bleu': 88.58227155698519}

================ Executing: nllb_zero_shot_aug=False ================


Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

Map:   0%|          | 0/19 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Bleu
No log,1.163305,0,7.070332


Completed! Metrics: {'eval_bleu': 7.070331532786434}


In [19]:
for strategy in strategies:

    exp = experiments["nllb"][strategy]

    nllb_trainer.predict_and_display(
        strategy=strategy,
        artifact_path=exp["artifact_path"],
        checkpoint_type=exp["checkpoint_type"],
        dataset=validation_data,
        num_samples=10
    )

    flush_gpu_memory()

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]


================ Sample Validation Predictions (facebook/nllb-200-1.3B) ================
Strategy       : freeze_encoder
Checkpoint     : delta
Validation BLEU: 14.02


,Source Text,Ground Truth,Model Prediction,BLEU Score
0,All belong! Enroll refugee children in schools today.,Wasajili watoto wakimbizi shuleni leo.,Watoto wa wakimbizi wametumwa katika shule leo.,14.01894
1,"All boarding schools must conduct safety audit by September 30, 2025.","Shule zote za bweni lazima zifanye ukaguzi wa usalama ifikapo Septemba 30, 2025.","Shule zote za kudumu lazima zimefanya ukeketaji kabla ya Septemba 30, 2025.",14.01894
2,All boarding schools must install fire alarms and sprinklers by January 6 CS orders!,Shule zote za bweni lazima ziweke kengele za moto na vinyunyiziaji maji kabla ya amri ya Waziri (CS) ya Januari 6!,Shule zote za kudumu zimepasa kuhama bila malipo na vimeji lazima vimeweka bila malipo bila malipo.,14.01894
3,All can learn! Use braille and ramps for disabled students.,Kila mtu anaweza kujifunza! Tumia maandishi ya nukta nundu (Braille) na njia panda (ramps) kwa ajili ya wanafunzi wenye ulemavu.,Kila mtu anaweza kujifunza!,14.01894
4,All dormitories must have emergency exits. Inspection starts December 1!,Kila bweni linapaswa kuwa na mlango wa kutokea wakati wa dharura.,Vyumba vyote vya ushirika lazima vimepata njia za dharura.,14.01894
5,All Form 3 students must register as voters this week. No excuses!,Wanafunzi wote wa Kidato cha 3 lazima wasajiliwe kuwa wapiga kura wiki hii.,Wanafunzi wote wa Gredi ya 3 lazima wasajili kama wapigaji wiki hii.,14.01894
6,All Form Four students must register as voters before leaving school.,Wanafunzi wote wa Kidato cha 4 lazima wasajiliwe kuwa wapiga kura kabla ya kumaliza shule.,Wanafunzi wote wa Form ya 4 lazima wasajili kama wapigaji kabla ya kuacha shule.,14.01894
7,All Grade 6 learners automatically placed in junior secondary no exam needed.,Wanafunzi wote wa Gredi ya 6 watapelekwa moja kwa moja katika shule ya upili ya chini (JSS) bila kuhitaji mtihani.,"Walimu wote wa Gredi ya 6 wametumwa bila malipo katika shule za upili, hakuna mtihani wowote unaohitajika.",14.01894
8,All public university students to get free data bundles for e-learning.,Wanafunzi wote wa vyuo vikuu vya umma watapata data (bundles) bila malipo kwa ajili ya kujifunza mtandaoni.,Wanafunzi wote wa chuo cha umma wametakiwa bila malipo bila malipo.,14.01894


Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]


================ Sample Validation Predictions (facebook/nllb-200-1.3B) ================
Strategy       : lora
Checkpoint     : lora
Validation BLEU: 40.87


,Source Text,Ground Truth,Model Prediction,BLEU Score
0,All belong! Enroll refugee children in schools today.,Wasajili watoto wakimbizi shuleni leo.,Wote wanastahili! Wasajili watoto wakimbizi shuleni leo.,40.872821
1,"All boarding schools must conduct safety audit by September 30, 2025.","Shule zote za bweni lazima zifanye ukaguzi wa usalama ifikapo Septemba 30, 2025.","Shule zote za bweni lazima zifanye ukaguzi wa usalama ifikapo Septemba 30, 2025.",40.872821
2,All boarding schools must install fire alarms and sprinklers by January 6 CS orders!,Shule zote za bweni lazima ziweke kengele za moto na vinyunyiziaji maji kabla ya amri ya Waziri (CS) ya Januari 6!,Shule zote za bweni lazima ziweke kengele za moto na vimumunyisho kabla ya maagizo ya CS ya Januari 6!,40.872821
3,All can learn! Use braille and ramps for disabled students.,Kila mtu anaweza kujifunza! Tumia maandishi ya nukta nundu (Braille) na njia panda (ramps) kwa ajili ya wanafunzi wenye ulemavu.,Wote wanaweza kujifunza! Tumia maandishi ya vipofu na rampe kwa wanafunzi wenye ulemavu.,40.872821
4,All dormitories must have emergency exits. Inspection starts December 1!,Kila bweni linapaswa kuwa na mlango wa kutokea wakati wa dharura.,Makao yote ya kulala lazima yawe na njia za kuondoka za dharura.,40.872821
5,All Form 3 students must register as voters this week. No excuses!,Wanafunzi wote wa Kidato cha 3 lazima wasajiliwe kuwa wapiga kura wiki hii.,Wanafunzi wote wa Fomu ya 3 lazima wajisajili kama wapiga kura wiki hii.,40.872821
6,All Form Four students must register as voters before leaving school.,Wanafunzi wote wa Kidato cha 4 lazima wasajiliwe kuwa wapiga kura kabla ya kumaliza shule.,Wanafunzi wote wa Fomu ya Nne lazima wasajili kama wapiga kura kabla ya kuacha shule.,40.872821
7,All Grade 6 learners automatically placed in junior secondary no exam needed.,Wanafunzi wote wa Gredi ya 6 watapelekwa moja kwa moja katika shule ya upili ya chini (JSS) bila kuhitaji mtihani.,Wanafunzi wote wa Daraja la 6 huwekwa moja kwa moja katika shule za upili za ujana hakuna mtihani unaohitajika.,40.872821
8,All public university students to get free data bundles for e-learning.,Wanafunzi wote wa vyuo vikuu vya umma watapata data (bundles) bila malipo kwa ajili ya kujifunza mtandaoni.,Wanafunzi wote wa vyuo vikuu vya umma wapate vifurushi vya data bila malipo kwa ujifunzaji wa kielektroniki.,40.872821


Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]


================ Sample Validation Predictions (facebook/nllb-200-1.3B) ================
Strategy       : zero_shot
Checkpoint     : base
Validation BLEU: 30.20


,Source Text,Ground Truth,Model Prediction,BLEU Score
0,All belong! Enroll refugee children in schools today.,Wasajili watoto wakimbizi shuleni leo.,Kujiandikisha watoto wakimbizi katika shule leo.,30.198216
1,"All boarding schools must conduct safety audit by September 30, 2025.","Shule zote za bweni lazima zifanye ukaguzi wa usalama ifikapo Septemba 30, 2025.","Shule zote za bweni lazima zifanye ukaguzi wa usalama kabla ya Septemba 30, 2025.",30.198216
2,All boarding schools must install fire alarms and sprinklers by January 6 CS orders!,Shule zote za bweni lazima ziweke kengele za moto na vinyunyiziaji maji kabla ya amri ya Waziri (CS) ya Januari 6!,Shule zote za bweni lazima ziweke tahadhari za moto na vipasha maji kabla ya Januari 6 amri za CS!,30.198216
3,All can learn! Use braille and ramps for disabled students.,Kila mtu anaweza kujifunza! Tumia maandishi ya nukta nundu (Braille) na njia panda (ramps) kwa ajili ya wanafunzi wenye ulemavu.,Kila mtu anaweza kujifunza! Tumia maandishi ya vipofu na rampe kwa wanafunzi wenye ulemavu.,30.198216
4,All dormitories must have emergency exits. Inspection starts December 1!,Kila bweni linapaswa kuwa na mlango wa kutokea wakati wa dharura.,Mahali pote pa kulala lazima pawe na njia za kutoka.,30.198216
5,All Form 3 students must register as voters this week. No excuses!,Wanafunzi wote wa Kidato cha 3 lazima wasajiliwe kuwa wapiga kura wiki hii.,Wanafunzi wote wa darasa la tatu lazima wajiandikishe kama wapiga kura wiki hii.,30.198216
6,All Form Four students must register as voters before leaving school.,Wanafunzi wote wa Kidato cha 4 lazima wasajiliwe kuwa wapiga kura kabla ya kumaliza shule.,Wanafunzi wote wa Kiwango cha Nne lazima waandikishwe kama wapiga kura kabla ya kuondoka shuleni.,30.198216
7,All Grade 6 learners automatically placed in junior secondary no exam needed.,Wanafunzi wote wa Gredi ya 6 watapelekwa moja kwa moja katika shule ya upili ya chini (JSS) bila kuhitaji mtihani.,Wanafunzi wote wa darasa la 6 moja kwa moja kuwekwa katika sekondari ya chini hakuna mtihani zinahitajika.,30.198216
8,All public university students to get free data bundles for e-learning.,Wanafunzi wote wa vyuo vikuu vya umma watapata data (bundles) bila malipo kwa ajili ya kujifunza mtandaoni.,Wanafunzi wote wa chuo kikuu cha umma kupata vifurushi vya data bure kwa e-kujifunza.,30.198216


From the examples comparing the `freeze_encoder` and `LoRA` translations it is clear that the `freeze_encoder` strategy overfits and hits a bottleneck that is not evident in `LoRA` to this extent. However, LoRA also overfits since zero shot performs a singnificant degree better.

In [20]:
flush_gpu_memory()

In [21]:
mbart_trainer = MBARTSeq2SeqTrainer(
    model_name="facebook/mbart-large-50-many-to-many-mmt",
    experiment_name="nllb_vs_mbart_ablations",
    src_lang="en_XX",
    tgt_lang="sw_KE"
)

In [22]:
for strategy in strategies:

    experiments["mbart"][strategy] = mbart_trainer.run_experiment(
        dataset=train_data,
        tuning_strategy=strategy,
        epochs=EPOCHS,
        lr=5e-4
    )

    flush_gpu_memory()


================ mbart_freeze_encoder_aug=False ================


[transformers] Could not extract SentencePiece model from /home/jovyan/.cache/huggingface/hub/models--facebook--mbart-large-50-many-to-many-mmt/snapshots/e30b6cb8eb0d43a0b73cab73c7676b9863223a30/sentencepiece.bpe.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.

In [ ]:
for strategy in strategies:
  exp = experiments["mbart"][strategy]

  mbart_trainer.predict_and_display(
      strategy=strategy,
      artifact_path=exp["artifact_path"],
      checkpoint_type=exp["checkpoint_type"],
      dataset=validation_data,
      num_samples=10
  )

  flush_gpu_memory()

In [ ]:
flush_gpu_memory()

In [ ]:
exp = mlflow.get_experiment_by_name("nllb_vs_mbart_ablations")
runs_df = mlflow.search_runs(experiment_ids=[exp.experiment_id])

In [ ]:
os.makedirs("reports", exist_ok=True)

In [ ]:
cols = [
    "tags.mlflow.runName",
    "params.model_name",
    "params.tuning_strategy",
    "params.use_augmentation",
    "params.learning_rate",
    "params.epochs",
    "params.lora_r",
    "params.src_lang",
    "params.tgt_lang",
    "params.checkpoint_type",
    "metrics.eval_bleu",
    "metrics.train_loss",
    "start_time",
    "end_time",
    "status"
]

In [ ]:
summary_df = (
    runs_df[[c for c in cols if c in runs_df.columns]]
    .rename(columns={
        "tags.mlflow.runName": "Run",
        "params.model_name": "Model",
        "params.tuning_strategy": "Strategy",
        "params.use_augmentation": "Augmented",
        "params.learning_rate": "Learning Rate",
        "params.epochs": "Epochs",
        "params.lora_r": "LoRA Rank",
        "params.src_lang": "Source Language",
        "params.tgt_lang": "Target Language",
        "params.checkpoint_type": "Checkpoint",
        "metrics.eval_bleu": "BLEU",
        "metrics.train_loss": "Train Loss",
        "status": "Status"
    })
)

summary_df["Model"] = summary_df["Model"].replace({
    "facebook/nllb-200-1.3B": "NLLB-200",
    "facebook/mbart-large-50-many-to-many-mmt": "mBART-50"
})

summary_df = summary_df.sort_values("BLEU", ascending=False).reset_index(drop=True)

summary_df.to_csv(
    "reports/experiment_summary.csv",
    index=False
)

In [ ]:
display(summary_df)

In [ ]:
best = summary_df.loc[summary_df["BLEU"].idxmax()]

print("\n========== BEST EXPERIMENT ==========\n")

print(f"Run             : {best['Run']}")
print(f"Model           : {best['Model']}")
print(f"Strategy        : {best['Strategy']}")
print(f"BLEU            : {best['BLEU']:.2f}")
print(f"Training Loss   : {best['Train Loss']:.4f}")
print(f"Learning Rate   : {best['Learning Rate']}")
print(f"Epochs          : {best['Epochs']}")
print(f"Augmentation    : {best['Augmented']}")
print(f"Checkpoint Type : {best['Checkpoint']}")


In [ ]:
best_per_model = (
    summary_df
    .sort_values("BLEU", ascending=False)
    .groupby("Model", as_index=False)
    .first()
)

best_per_model.to_csv(
    "reports/best_per_model.csv",
    index=False
)

In [ ]:
display(best_per_model)

In [ ]:
strategy_summary = (
    summary_df
    .groupby(["Model", "Strategy"])
    .agg({
        "BLEU": "mean",
        "Train Loss": "mean"
    })
    .round(3)
    .reset_index()
)

strategy_summary.to_csv(
    "reports/strategy_summary.csv",
    index=False
)

In [ ]:
display(strategy_summary)

In [ ]:
pivot = (
    summary_df
    .pivot_table(
        index="Strategy",
        columns="Model",
        values="BLEU",
        aggfunc="mean"
    )
    .round(2)
)

pivot.to_csv(
    "reports/bleu_pivot.csv"
)

In [ ]:
display(pivot)

In [ ]:
files.download("reports/experiment_summary.csv")
files.download("reports/best_per_model.csv")
files.download("reports/strategy_summary.csv")
files.download("reports/bleu_pivot.csv")